# Markov Network Basics: Construction and Inspection

This notebook covers the fundamentals of working with Markov networks
(undirected graphical models) in conin: defining states, building factor
potentials, assembling a network, and inspecting its structure.

See also the [constraints](constraints.ipynb) and
[inference](inference.ipynb) notebooks for how to add constraints and run
MAP queries on Markov networks.

In [ ]:
from conin.markov_network import DiscreteMarkovNetwork, DiscreteFactor

## Defining States

Like Bayesian networks, a Markov network starts with a set of nodes and
their possible states.

In [ ]:
mn = DiscreteMarkovNetwork()
mn.states = {"A": [0, 1, 2], "B": [0, 1, 2], "C": [0, 1, 2]}

print("Nodes:      ", sorted(mn.states.keys()))
print("States of A:", mn.states_of("A"))
print("Card of B:  ", mn.card("B"))

## Building Factors

A Markov network is parameterized by **factors** (also called potentials).
Each factor is a non-negative function over a subset of nodes. A
`DiscreteFactor` takes:

- `nodes` — the variables in the factor's scope
- `values` — a dict mapping state assignments to non-negative weights

Unlike CPDs in a Bayesian network, factor values do not need to sum to 1;
they represent relative weights.

### Unary factors

A factor over a single node assigns a weight to each of that node's states.

In [ ]:
f_a = DiscreteFactor(nodes=["A"], values={0: 1, 1: 1, 2: 2})
f_b = DiscreteFactor(nodes=["B"], values={0: 1, 1: 1, 2: 3})
f_c = DiscreteFactor(nodes=["C"], values={0: 1, 1: 2, 2: 1})

### Pairwise factors

A factor over two nodes assigns a weight to each pair of states. Keys are
tuples in the same order as the `nodes` list.

In [ ]:
# Uniform pairwise factors — all pairs get weight 1
f_ab = DiscreteFactor(
    nodes=["A", "B"],
    values={(i, j): 1 for i in range(3) for j in range(3)},
)
f_bc = DiscreteFactor(
    nodes=["B", "C"],
    values={(i, j): 1 for i in range(3) for j in range(3)},
)
f_ac = DiscreteFactor(
    nodes=["A", "C"],
    values={(i, j): 1 for i in range(3) for j in range(3)},
)

## Assembling the Network

Assign factors via the `factors` property. Edges are inferred from factor
scopes automatically — any factor over two or more nodes introduces edges
between them.

In [ ]:
mn.factors = [f_a, f_b, f_c, f_ab, f_bc, f_ac]

mn.check_model()
print("Model is valid.")
print("Edges:", sorted(mn.edges))

## Inspecting the Network

In [ ]:
for f in mn.factors:
    print(f"Factor({f.nodes}):")
    for key, val in f.values.items():
        print(f"  {key}: {val}")

## A Minimal Example

States, edges, and factors can all be passed to the constructor.

In [ ]:
simple_mn = DiscreteMarkovNetwork(
    states={"X": [0, 1], "Y": [0, 1]},
    factors=[
        DiscreteFactor(nodes=["X"], values={0: 1, 1: 2}),
        DiscreteFactor(nodes=["Y"], values={0: 1, 1: 3}),
        DiscreteFactor(
            nodes=["X", "Y"],
            values={(0, 0): 1, (0, 1): 2, (1, 0): 3, (1, 1): 1},
        ),
    ],
)
simple_mn.check_model()

print("Edges: ", simple_mn.edges)
print("States:", simple_mn.states)